# Feature Engineering (Train/Test)
Create the same engineered features on both train and test datasets.

In [ ]:
import pandas as pd

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Standardize raw string columns
    for col in out.select_dtypes(include='object').columns:
        out[col] = out[col].astype(str).str.strip()

    # Convert numeric-like columns
    out['tenure'] = pd.to_numeric(out['tenure'], errors='coerce')
    out['MonthlyCharges'] = pd.to_numeric(out['MonthlyCharges'], errors='coerce')
    if 'TotalCharges' in out.columns:
        out['TotalCharges'] = pd.to_numeric(out['TotalCharges'], errors='coerce')
        out['TotalCharges'] = out['TotalCharges'].fillna(out['MonthlyCharges'] * out['tenure'].fillna(0))

    out['tenure_group'] = pd.cut(
        out['tenure'].fillna(-1),
        bins=[-1, 12, 48, 120],
        labels=['new', 'mid', 'loyal']
    )

    service_cols = [
        'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]

    # Chỉ tính dịch vụ đang active là Yes
    for col in service_cols:
        out[col] = out[col].fillna('No')
    out['num_services'] = (out[service_cols] == 'Yes').sum(axis=1)

    out['fiber_month_to_month'] = (
        (out['InternetService'] == 'Fiber optic') &
        (out['Contract'] == 'Month-to-month')
    ).astype(int)

    out['monthly_by_tenure'] = out['MonthlyCharges'] / out['tenure'].clip(lower=1)
    return out

train_fe = add_features(train_df)
test_fe = add_features(test_df)

# Schema check to keep train/test pipeline consistent
train_feature_cols = sorted([c for c in train_fe.columns if c not in ['Churn']])
test_feature_cols = sorted(test_fe.columns.tolist())
if train_feature_cols != test_feature_cols:
    missing_in_test = sorted(set(train_feature_cols) - set(test_feature_cols))
    extra_in_test = sorted(set(test_feature_cols) - set(train_feature_cols))
    raise ValueError(f'Schema mismatch. Missing in test: {missing_in_test}; Extra in test: {extra_in_test}')

print('train_fe shape:', train_fe.shape)
print('test_fe shape :', test_fe.shape)
print('Missing values (train_fe, top 10):')
print(train_fe.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
display(train_fe[['tenure_group', 'num_services', 'fiber_month_to_month', 'monthly_by_tenure', 'Churn']].head())
display(test_fe[['tenure_group', 'num_services', 'fiber_month_to_month', 'monthly_by_tenure']].head())

In [ ]:
train_fe.to_csv('../data/train_fe.csv', index=False)
test_fe.to_csv('../data/test_fe.csv', index=False)
print('Saved ../data/train_fe.csv and ../data/test_fe.csv')